# create sepecfic dataset folders for adni aibl and oasis

copy clinical and biomarkers data in csv or excel format

create a manifest file for each dataset

no data leaks: only demographics or counts are printed

In [2]:
from pathlib import Path
import csv
import hashlib
import shutil
import zipfile
import xml.etree.ElementTree as ET

project_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'SkullStrippingProject']
CODE_ROOT = next(path for path in project_candidates if (path / 'src' / 'skullstrip_pipeline').exists())
PROJECT_ROOT = CODE_ROOT.parent
DEST_ROOT = PROJECT_ROOT / 'Processed' / 'Clinical'
DATASET_ROOTS = {
    'adni': PROJECT_ROOT / 'ADNI',
    'aibl': PROJECT_ROOT / 'AIBL',
    'oasis': PROJECT_ROOT / 'OASIS',
}
for dataset in DATASET_ROOTS:
    (DEST_ROOT / dataset).mkdir(parents=True, exist_ok=True)

def is_data_table(path):
    return path.suffix.lower() in {'.csv', '.xls', '.xlsx'}

def selected_files(dataset, root):
    selected = []
    for path in root.rglob('*'):
        if not path.is_file() or not is_data_table(path):
            continue
        rel = path.relative_to(root)
        text = str(rel).lower()
        name = path.name.lower()
        if dataset == 'adni':
            clinical_dirs = 'tables' in {part.lower() for part in rel.parts}
            clinical_name = any(token in name for token in (
                'adni1_', 'customids', 'study_entry', 'lab', 'mmse',
                'medhist', 'physical', 'faq', 'vitals', 'mrifind',
                'neuropath', 'biomarker', 'harp_data'))
            keep = (clinical_dirs or clinical_name) and 'reports' not in text
        elif dataset == 'aibl':
            keep = 'macosx' not in text and 'data_extract_3.3.0' in text
        else:
            oasis2 = name == 'oasis_2.csv'
            oasis3_clinical = ('oasis3_data_files/scans/' in text and any(token in text for token in (
                'uds', 'demo-demographics', 'participant_demo', 'braak',
                'centiloid', 'amyloid')))
            local_master = text.endswith('/oasis_master_dataset.csv')
            keep = oasis2 or oasis3_clinical or local_master
        if keep:
            selected.append(path)
    return sorted(selected)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def csv_shape(path):
    with path.open('r', encoding='utf-8-sig', errors='replace', newline='') as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration:
            return 0, 0
        rows = sum(1 for _ in reader)
    return rows, len(header)

def table_shape(path):
    if path.suffix.lower() == '.csv':
        return csv_shape(path)
    return None, None

def copy_dataset(dataset, source_root):
    destination = DEST_ROOT / dataset
    records = []
    for source in selected_files(dataset, source_root):
        relative = source.relative_to(source_root)
        copied = destination / relative
        copied.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, copied)
        rows, columns = table_shape(copied)
        records.append({
            'dataset': dataset,
            'source_relative_path': str(relative),
            'copied_relative_path': str(relative),
            'file_type': copied.suffix.lower().lstrip('.'),
            'size_bytes': copied.stat().st_size,
            'sha256': sha256(copied),
            'row_count': rows if rows is not None else '',
            'column_count': columns if columns is not None else '',
        })
    manifest = destination / f'{dataset}_clinical_manifest.csv'
    fields = ['dataset', 'source_relative_path', 'copied_relative_path', 'file_type', 'size_bytes', 'sha256', 'row_count', 'column_count']
    with manifest.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    return records

results = {dataset: copy_dataset(dataset, root) for dataset, root in DATASET_ROOTS.items()}
print('copy complete')


copy complete


## output summary

only file counts, table row counts, and table column counts are printed. no subject ids or clinical values are printed.

In [2]:
for dataset, records in results.items():
    total_rows = sum(int(record['row_count']) for record in records if record['row_count'] != '')
    csv_files = sum(record['file_type'] == 'csv' for record in records)
    excel_files = sum(record['file_type'] in {'xls', 'xlsx'} for record in records)
    print(f'{dataset}: files={len(records)}, csv={csv_files}, excel={excel_files}, rows={total_rows}')


adni: files=15, csv=15, excel=0, rows=102998
aibl: files=18, csv=18, excel=0, rows=17286
oasis: files=46, csv=44, excel=2, rows=292659


## environment note

this notebook uses only the Python standard library so it can run in the project environment without exposing clinical rows.